In [ ]:
!pip install wikipedia

In [ ]:
import wikipedia
import re

wikipedia.set_lang("en")

def fetch_wikipedia_page(title):
    try:
        # 🔥 先搜索
        results = wikipedia.search(title)
        
        if not results:
            print(f"❌ No results for {title}")
            return None
        
        # 🔥 取第一个最相关的
        page_title = results[0]
        
        page = wikipedia.page(page_title, auto_suggest=False)
        
        text = page.content
        
        # 清洗
        text = re.sub(r"\[\d+\]", "", text)
        text = re.sub(r"\n{2,}", "\n", text)
        text = re.sub(r"[ \t]+", " ", text)
        
        return {
            "title": page.title,
            "text": text.strip()
        }
    
    except Exception as e:
        print(f"❌ Error fetching {title}: {e}")
        return None

In [ ]:
import requests
import time

def get_all_titles(category):
    url = "https://en.wikipedia.org/w/api.php"
    
    titles = []
    cmcontinue = None

    while True:
        params = {
            "action": "query",
            "format": "json",
            "list": "categorymembers",
            "cmtitle": f"Category:{category}",
            "cmlimit": 500
        }

        if cmcontinue:
            params["cmcontinue"] = cmcontinue

        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        try:
            res = requests.get(url, params=params, headers=headers)

            # 防止被限流
            if "application/json" not in res.headers.get("Content-Type", ""):
                print("⚠️ Rate limited, sleeping...")
                time.sleep(3)
                continue

            data = res.json()

            titles += [item["title"] for item in data["query"]["categorymembers"]]

            if "continue" in data:
                cmcontinue = data["continue"]["cmcontinue"]
            else:
                break

        except Exception as e:
            print(f"❌ Error in {category}: {e}")
            break

        time.sleep(0.5)

    print(f"✅ {category}: {len(titles)} items")
    return titles

In [ ]:

# 自动获取菜名
categories = [
    # cuisine
    "Chinese cuisine",
    "Japanese cuisine",
    "Korean cuisine",

    # chinese补充（关键）
    "Chinese noodles",
    "Chinese rice dishes",
    "Chinese dumplings",
    "Chinese soups",

    # japanese / korean dishes（可选）
    "Japanese dishes",
    "Korean dishes"
]

topics = []

for cat in categories:
    print(f"Fetching category: {cat}")
    
    try:
        titles = get_all_titles(cat)
        
        if not titles:
            print(f"⚠️ Empty result for {cat}")
            continue
        
        topics.extend(titles)
        print(f"✅ {cat}: {len(titles)} items")

    except Exception as e:
        print(f"❌ Error in {cat}: {e}")
        continue

# 去重
topics = list(set(topics))

print(f"\n🎯 Total topics: {len(topics)}")

In [ ]:
bad_keywords = [
    "chef", "restaurant", "company", "brand",
    "cookbook", "tv", "film", "list", "cuisine"
]

clean_topics = []

for t in topics:
    t_lower = t.lower()

    if t.startswith("Category:"):
        continue
    
    if any(k in t_lower for k in bad_keywords):
        continue

    clean_topics.append(t)

topics = clean_topics

In [ ]:
corpus = []

for i, topic in enumerate(topics):

    if topic.startswith("Category:"):
        continue

    if i % 30 == 0:
        print("🛑 Cooling down...")
        time.sleep(10)
        
    print(f"Fetching: {topic}")

    data = fetch_wikipedia_page(topic)

    if data and data["text"]:
        corpus.append({
            "doc_id": f"doc_{abs(hash(topic))}",
            "title": data["title"],
            "text": data["text"]
        })

    time.sleep(1.5)  # ⭐ 非常重要